# ICFES ETL, Multi-period profiling and schema control

This notebook profiles and consolidates Saber 11 results from DataIcfes (flat
.txt files) for periods 2021-2 through 2025-2, accounting for the fact
that not every period contains exactly the same columns.

Because the real files are large (roughly tens to hundreds of MB, with
hundreds of thousands of rows), the notebook does not load an entire file
into memory at once: it detects encoding/separator from a sample, profiles
headers only, and processes/consolidates data in chunks.

The notebook performs four main tasks:

1. Detects encoding and separator for each file and reads only the
   headers for schema profiling.
2. Profiles the schema of each period and records changes between
   consecutive periods.
3. Builds a master schema and profiles the source datasets while preserving
   complete row counts and null counts.
4. Profiles data quality for fields used by the dimensional model and
   project validation rules.

The notebook is used only for profiling and quality analysis. The ETL pipeline
in src/main.py reads the raw .txt files directly and loads the dimensional
warehouse in MySQL.


## 0. Configuration

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import csv
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Academic periods to process (year-semester)
PERIODOS = [
    "2021-2", "2022-1", "2022-2", "2023-1", "2023-2",
    "2024-1", "2024-2", "2025-1", "2025-2",
]

# Project paths
# Locate the project root from the notebook's current working directory.
# This works whether Jupyter was launched from the project root or notebooks/.
_CURRENT_DIR = Path.cwd().resolve()
_PROJECT_ROOT_CANDIDATES = [_CURRENT_DIR, *_CURRENT_DIR.parents]
PROJECT_ROOT = next(
    (p for p in _PROJECT_ROOT_CANDIDATES
     if (p / "data" / "raw").is_dir() and (p / "data" / "processed").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook inside the "
        "project so that data/raw and data/processed are available."
    )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Processed output directory: {OUTPUT_DIR}")

# Chunk size for reading/consolidating large source files
CHUNK_SIZE = 100_000

# Sample rows per period for descriptive statistics in section 4.3
FILAS_MUESTRA_POR_PERIODO = 20_000

# If the separator detector is wrong for a specific period, override it here.
# Example: SEPARADORES_MANUAL = {"2022-1": "|"}
SEPARADORES_MANUAL = {}

def periodo_a_nombre_archivo(periodo: str) -> str:
    """Convert '2021-2' to the actual DataIcfes filename pattern."""
    codigo = periodo.replace("-", "")
    return f"Examen_Saber_11_{codigo}.txt"

print(f"Periods to process ({len(PERIODOS)}): {PERIODOS}")


## 1. Encoding/separator detection and header inspection

The encoding and separator are detected from a small sample of each file.
Only the headers (nrows=0) are then read for schema profiling, so sections
1 and 2 remain fast even when the source files are large.


In [ ]:
def _detectar_encoding(archivo: Path, muestra_bytes: int = 200_000) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            with open(archivo, "r", encoding=enc) as f:
                f.read(muestra_bytes)
            return enc
        except UnicodeDecodeError:
            continue
    return "latin-1"  # fallback: latin-1

def _detectar_separador(archivo: Path, encoding: str) -> str:
    with open(archivo, "r", encoding=encoding, errors="replace") as f:
        sample = f.read(65_536)
    try:
        return csv.Sniffer().sniff(sample, delimiters="|;,\t¬").delimiter
    except csv.Error:
        first_line = sample.splitlines()[0] if sample else ""
        candidates = ["|", ";", ",", "\t", "¬"]
        return max(candidates, key=first_line.count)

def info_archivo_periodo(periodo: str) -> dict:
    """Locate the period file and detect its encoding/separator."""
    archivo = RAW_DATA_DIR / periodo_a_nombre_archivo(periodo)
    if not archivo.exists():
        raise FileNotFoundError(
            f"Source file not found: '{archivo}'. Put the DataIcfes .txt files in "
            f"data/raw/ or verify the project root was detected correctly."
        )
    encoding = _detectar_encoding(archivo)
    separador = SEPARADORES_MANUAL.get(periodo) or _detectar_separador(archivo, encoding)
    return {"archivo": archivo, "encoding": encoding, "separador": separador}

info_periodos = {periodo: info_archivo_periodo(periodo) for periodo in PERIODOS}
for periodo, info in info_periodos.items():
    print(f"[{periodo}] {info['archivo'].name} -> encoding={info['encoding']}, separator={info['separador']!r}")


In [ ]:
def leer_columnas(periodo: str) -> list:
    info = info_periodos[periodo]
    encabezado = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], nrows=0)
    return list(encabezado.columns)

columnas_por_periodo = {periodo: leer_columnas(periodo) for periodo in PERIODOS}
for periodo, cols in columnas_por_periodo.items():
    print(f"[{periodo}] {len(cols)} columnas")


## 2. Schema profiling by period

For each period (in chronological order), the set of columns is compared
with the immediately preceding period:

- columnas_nuevas: columns that appear now but were not present before.
- columnas_eliminadas: columns that were present before but are now missing.
- estado: OK when the schema is unchanged, CAMBIO when at least one
  column was added or removed.

The first period is used as the baseline (OK, with no previous period to
compare against).


In [ ]:
def construir_tabla_control(columnas_por_periodo: dict, periodos_ordenados: list) -> pd.DataFrame:
    filas = []
    columnas_anteriores = None
    for periodo in periodos_ordenados:
        columnas_actuales = set(columnas_por_periodo[periodo])

        if columnas_anteriores is None:
            nuevas, eliminadas, estado = set(), set(), "OK"
        else:
            nuevas = columnas_actuales - columnas_anteriores
            eliminadas = columnas_anteriores - columnas_actuales
            estado = "CAMBIO" if (nuevas or eliminadas) else "OK"

        filas.append({
            "periodo": periodo,
            "numero_columnas": len(columnas_actuales),
            "columnas_nuevas": ", ".join(sorted(nuevas)) if nuevas else "-",
            "columnas_eliminadas": ", ".join(sorted(eliminadas)) if eliminadas else "-",
            "fecha_carga": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "estado": estado,
        })
        columnas_anteriores = columnas_actuales

    return pd.DataFrame(filas)

etl_schema_control = construir_tabla_control(columnas_por_periodo, PERIODOS)
etl_schema_control


In [ ]:
# Summary view: period | number of columns | status
etl_schema_control[["periodo", "numero_columnas", "estado"]]


## 3. Schema harmonization

A master schema is built as the union of all columns observed across the
periods, preserving the order in which columns first appeared. Each period
is reindexed against this master schema in section 4, filling missing source
columns with NaN. This allows all periods to be consolidated without
discarding information when schemas differ.


In [ ]:
def calcular_esquema_maestro(columnas_por_periodo: dict, periodos_ordenados: list) -> list:
    maestro = []
    vistas = set()
    for periodo in periodos_ordenados:
        for col in columnas_por_periodo[periodo]:
            if col not in vistas:
                maestro.append(col)
                vistas.add(col)
    return maestro

esquema_maestro = calcular_esquema_maestro(columnas_por_periodo, PERIODOS)
print(f"Columnas en el esquema maestro: {len(esquema_maestro)}")


### 3.1 Column coverage matrix by period

For every period, this view shows which master-schema columns were actually
present in the original source file. It is useful for visually identifying
when the source schema changed.


In [ ]:
cobertura = pd.DataFrame(
    {periodo: [col in columnas_por_periodo[periodo] for col in esquema_maestro] for periodo in PERIODOS},
    index=esquema_maestro,
).T  # rows = periodo, columns = master schema variable

cobertura.replace({True: "x", False: ""})


## 4. Chunked profiling

The actual data is read here (not only headers), but in CHUNK_SIZE row
blocks so the full multi-million-row dataset is never loaded into memory at once.
Each block is reindexed against the master schema and receives the standardized
periodo value in YYYY-S format.

Exact row counts and null counts are accumulated across the complete source dataset.


In [ ]:
filas_por_periodo = {}
nulos_por_columna = pd.Series(0, index=esquema_maestro, dtype="int64")

for periodo in PERIODOS:
    info = info_periodos[periodo]
    total_filas_periodo = 0
    lector = pd.read_csv(
        info["archivo"], sep=info["separador"], encoding=info["encoding"],
        chunksize=CHUNK_SIZE,
    )
    for bloque in lector:
        bloque = bloque.reindex(columns=esquema_maestro)
        bloque["periodo"] = periodo  # standardize the period value to YYYY-S format
        nulos_por_columna += bloque[esquema_maestro].isna().sum()
        total_filas_periodo += len(bloque)
    filas_por_periodo[periodo] = total_filas_periodo
    print(f"[{periodo}] {total_filas_periodo:,} filas procesadas y perfiladas")

total_filas = sum(filas_por_periodo.values())
print(f"\nTotal de filas perfiladas: {total_filas:,}")


### 4.1 Records by period

In [ ]:
pd.Series(filas_por_periodo, name="filas").reindex(PERIODOS).to_frame()


### 4.2 Null percentage by column

Computed exactly over the complete source dataset by accumulating counts
chunk by chunk in the previous cell (not from a sample).


In [ ]:
pct_nulos = (nulos_por_columna / total_filas * 100).sort_values(ascending=False)
pct_nulos.round(1).to_frame("% nulos")


### 4.3 Descriptive statistics (sample-based)

With more than three million source rows, running describe() over the
complete consolidated dataset in pandas is more expensive than necessary for
an initial profile. This cell takes only the first
FILAS_MUESTRA_POR_PERIODO rows from each period to provide a quick view of
ranges and magnitudes. For exhaustive profiling of the complete consolidated
file, use a tool designed for large datasets (for example Dask, Polars,
DuckDB, or a database) rather than loading the entire CSV into pandas.


In [ ]:
muestras = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    m = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], nrows=FILAS_MUESTRA_POR_PERIODO)
    m = m.reindex(columns=esquema_maestro)
    m["periodo"] = periodo
    muestras.append(m)

df_muestra = pd.concat(muestras, ignore_index=True)
print(f"Muestra total: {df_muestra.shape[0]:,} filas, {df_muestra.shape[1]} columnas")
df_muestra.select_dtypes(include=["number"]).describe().T


In [ ]:
df_muestra.dtypes.to_frame("dtype")


## 4.4 Additional data-quality profiling

Focused checks that complement the general profiling above: duplicate natural
keys, estu_genero quality (used in R5), out-of-range punt_global values,
nulls in school attributes (used in R1-R4), and category inconsistencies in
cole_area_ubicacion.


Why compare by estu_consecutivo instead of the complete row? The goal
here is not to ask whether two rows are 100% identical across all columns.
The project is interested in whether the natural key value repeats: one
student-attempt-period. Two rows may share estu_consecutivo while differing
in another field (for example, a corrected or retransmitted record). A full
row-duplicate check would miss that situation, while the dimensional model
would still have two competing records for the same natural-key grain before
generating the fact-table surrogate key.


In [ ]:
# Duplicates by estu_consecutivo, by period
print("=== Duplicates by estu_consecutivo ===")
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["estu_consecutivo"])
    dup = d["estu_consecutivo"].duplicated().sum()
    print(f"{periodo}: {len(d):>7,} rows, {dup} duplicated consecutive IDs")


In [ ]:
# estu_genero quality (used in R5)
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["estu_genero"])
    partes.append(d)
genero = pd.concat(partes, ignore_index=True)
print("=== estu_genero ===")
print(genero["estu_genero"].value_counts(dropna=False))
print(f"% nulls: {genero['estu_genero'].isna().mean():.4%}")


In [ ]:
# Out-of-range values in punt_global (valid ICFES range: 0-500)
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["punt_global"], low_memory=False)
    partes.append(d)
pg = pd.concat(partes, ignore_index=True)["punt_global"]
print("=== punt_global ===")
print(f"min={pg.min()}, max={pg.max()}, nulls={pg.isna().sum()}")
print(f"records with punt_global == 0 (possible annulled/absent tests): {(pg == 0).sum()}")
print(f"records outside [0, 500]: {((pg < 0) | (pg > 500)).sum()}")


In [ ]:
# Nulls in school attributes (used in R1-R4)
cols_colegio = ["cole_naturaleza", "cole_jornada", "cole_area_ubicacion", "cole_depto_ubicacion", "cole_nombre_establecimiento"]
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=cols_colegio, low_memory=False)
    partes.append(d)
colegio = pd.concat(partes, ignore_index=True)

print("=== Nulls in school attributes (affect R1-R4) ===")
for c in cols_colegio:
    print(f"{c}: {colegio[c].isna().sum():,} nulls ({colegio[c].isna().mean():.2%})")

# Are the same rows null across both reference columns?
# This can indicate students without an associated school rather than capture errors.
mismas_filas = (colegio["cole_naturaleza"].isna() == colegio["cole_nombre_establecimiento"].isna()).mean()
print(f"\n% agreement between nulls in cole_naturaleza and cole_nombre_establecimiento: {mismas_filas:.2%}")
print("(A value close to 100% supports the interpretation that these are students without an associated school, e.g. independent test takers, rather than capture errors.)")


In [ ]:
# Category inconsistency in cole_area_ubicacion
partes = []
for periodo in PERIODOS:
    info = info_periodos[periodo]
    d = pd.read_csv(info["archivo"], sep=info["separador"], encoding=info["encoding"], usecols=["cole_area_ubicacion"], low_memory=False)
    partes.append(d)
area = pd.concat(partes, ignore_index=True)
print("=== cole_area_ubicacion values (note: URBANA / URBANO represent the same category with inconsistent labeling) ===")
print(area["cole_area_ubicacion"].value_counts(dropna=False))
